# 🎬 YT-Short-Clipper - Google Colab Edition
Automate high-quality viral shorts, reels, and TikToks creation from YouTube video links using AI highlights, auto face-cropping, and styling caption burns.

---

### ⚡ Cell 1: Verify Hardware Accelerator (GPU)
We recommend running this notebook on a **GPU T4** or better to speed up FFmpeg encoding and active speaker video cropping.

In [ ]:
!nvidia-smi
import torch
print("GPU Available:", torch.cuda.is_available())

### 📦 Cell 2: Install Dependencies & Clone Repository
Clones the official repository and downloads required system libraries (`ffmpeg`, `yt-dlp`, `mediapipe`, `openai`, etc.).

In [ ]:
# 1. Clone repository
!git clone https://github.com/amfajar/youtube-short-clipper /content/yt-short-clipper

# 2. Install requirements
%cd /content/yt-short-clipper
!pip install -r requirements.txt
!pip install google-genai openai torch requests

### ⚙️ Cell 3: Interactive Configuration Form
Input your YouTube URL, AI API keys, and clipping configurations below.

In [ ]:
#@title 🛠️ Clipper Settings { display-mode: "form" }

YOUTUBE_URL = "" #@param {type:"string"}
AI_PROVIDER = "openrouter" #@param ["openrouter", "google", "groq", "openai"]
API_KEY = "" #@param {type:"string"}
MODEL_NAME = "auto" #@param {type:"string"}

#@markdown ---
#@markdown #### 🎙️ Caption & Whisper Settings (Optional)
WHISPER_PROVIDER = "openai" #@param ["openai", "groq", "openrouter"]
WHISPER_API_KEY = "" #@param {type:"string"}

#@markdown ---
#@markdown #### 🎞️ Render Style
NUM_CLIPS = 5 #@param {type:"slider", min:1, max:10, step:1}
SUBTITLE_LANGUAGE = "id" #@param ["id", "en", "none"]
FACE_TRACKING_MODE = "opencv" #@param ["opencv", "mediapipe"]
ADD_CAPTIONS = True #@param {type:"boolean"}
ADD_HOOK = True #@param {type:"boolean"}

config_dict = {
    "url": YOUTUBE_URL,
    "provider": AI_PROVIDER,
    "api_key": API_KEY,
    "model": MODEL_NAME,
    "whisper_provider": WHISPER_PROVIDER,
    "whisper_api_key": WHISPER_API_KEY,
    "num_clips": NUM_CLIPS,
    "subtitle_language": SUBTITLE_LANGUAGE,
    "face_mode": FACE_TRACKING_MODE,
    "add_captions": ADD_CAPTIONS,
    "add_hook": ADD_HOOK
}
print("✅ Configuration saved programmatically.")

### 🍪 Cell 4: YouTube Cookie Verification (Optional)
Run this cell if the video is restricted, private, or causing `HTTP 403 Forbidden` errors. Click "Choose Files" and select your exported `cookies.txt` file.

In [ ]:
from google.colab import files
import os

print("Upload cookies.txt file if needed:")
uploaded = files.upload()
for filename in uploaded.keys():
    if filename == "cookies.txt":
        os.rename(filename, "/content/yt-short-clipper/cookies.txt")
        print("✅ cookies.txt uploaded and configured successfully!")
    else:
        os.remove(filename)
        print(f"❌ Ignored {filename}. Please upload a file named exactly cookies.txt")

### 🚀 Cell 5: RUN CLIPPING PIPELINE
Trigger the execution. Follow the visual logs below to track current progress.

In [ ]:
import sys
sys.path.insert(0, '/content/yt-short-clipper')

from colab_runner import run_colab_pipeline

output_files = run_colab_pipeline(config_dict)

### 📥 Cell 6: Download Short Clips
Zips all finished master videos inside `/content/output/` and prompts a local browser download.

In [ ]:
import shutil
from google.colab import files
import os

output_dir = "/content/output"
zip_path = "/content/yt_short_clips_export"

if os.path.exists(output_dir) and len(os.listdir(output_dir)) > 1:
    print("📦 Packaging clips into a zip archive...")
    shutil.make_archive(zip_path, 'zip', output_dir)
    
    zip_file = zip_path + ".zip"
    if os.path.exists(zip_file):
        print(f"📥 Downloading packed archive ({os.path.getsize(zip_file)/(1024*1024):.1f} MB)...")
        files.download(zip_file)
    else:
        print("❌ Failed to build zip archive.")
else:
    print("❌ No generated clips found in output folder. Ensure Cell 5 completed successfully.")